In [47]:
import sys
import os

# Manually specify the path to your project folder
project_path = os.path.abspath('..')  # Or provide the full path if needed
sys.path.append(project_path)

from helper import fl_validation
from models.model_utils import get_model
from run_intro_exp import calculate_distance_matrix
import torch
import torch.nn.functional as F
import torchvision.datasets as datasets
import torchvision.transforms as T
from functools import partial
import torch.nn as nn

def evaluate_models_and_compute_distances(
    pt_file_paths,
    model_fn,
    criterion,
    test_data,
    device='cpu',
    class_num=10,
    mali_labels=[0, 7],
    server_batch_size=32
):
    """
    Loads models from pt files, validates them, and computes distance matrices.
    
    Args:
        pt_file_paths (list): List of paths to .pt model weight files.
        model_fn (callable): Function that returns a fresh instance of the model.
        criterion: Loss function used for validation.
        test_data: Validation dataset.
        device (str): Device to run validation on.
        class_num (int): Number of classes.
        mali_labels (list): Labels for potential attack evaluation.
        server_batch_size (int): Batch size for validation.

    Returns:
        results (list): List of tuples (accuracy, per_class_accuracy, attack_success_rate).
        cos_diff_matrix (Tensor): Pairwise cosine distance matrix.
        l2_diff_matrix (Tensor): Pairwise L2 distance matrix.
    """
    flattened_updates = []
    results = []

    for pt_path in pt_file_paths:
        # Load model and weights
        model = model_fn().to(device)
        state_dict = torch.load(pt_path, map_location=device, weights_only=True)
        model.load_state_dict(state_dict)

        # Validation
        test_loss, accuracy, per_class_accuracy, attack_success_rate = fl_validation(
            model, criterion, test_data,
            server_batch_size=server_batch_size,
            class_num=class_num,
            attack_method='none',
            mali_labels=mali_labels,
            device=device
        )
        results.append(accuracy)

        # Flatten model parameters
        flat_params = flatten_model(model.state_dict())
        flattened_updates.append(flat_params)

    # Compute distance matrices
    cos_diff_matrix, l2_diff_matrix = calculate_distance_matrix(flattened_updates)

    return results, cos_diff_matrix, l2_diff_matrix

def flatten_model(state_dict):
    return torch.cat([param.flatten() for param in state_dict.values()])

In [48]:
transform_img = T.Compose([
    T.ToTensor(),
    T.Normalize((0.5,), (0.5,))
])

test_data = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_img)

Files already downloaded and verified


In [49]:
# pt_file_paths = ("/home2/liyang/GitHub/FedOda/experiments/20250510_173111_cifar10_MobileNetV2/model0.pt",
# "/home2/liyang/GitHub/FedOda/experiments/20250510_173111_cifar10_MobileNetV2/modelt.pt",
# "/home2/liyang/GitHub/FedOda/experiments/20250510_173111_cifar10_MobileNetV2/model_att.pt")
target = "/home2/liyang/GitHub/FedOda/experiments/20250511_125628_cifar10_ResNet18"
pt_file_paths = (f"{target}/model0.pt",
                f"{target}/modelt.pt",
                f"{target}/model_att.pt")

model_class = get_model('ResNet18')
model_fn = partial(model_class, num_classes=10, dataset="cifar10")
criterion = nn.CrossEntropyLoss()

In [50]:
results, cos_diff_matrix, l2_diff_matrix = evaluate_models_and_compute_distances(
    pt_file_paths,
    model_fn,
    criterion,
    test_data,
    device='cpu',
    class_num=10,
    mali_labels=[0, 7],
    server_batch_size=32
)

In [51]:
results

[0.1, 0.553, 0.1]

In [53]:
l2_diff_matrix

tensor([[0.0000e+00, 1.4311e+04, 1.4311e+04],
        [1.4311e+04, 0.0000e+00, 2.8573e+00],
        [1.4311e+04, 2.8573e+00, 0.0000e+00]])

In [ ]:
cos_diff_matrix

tensor([[ 0.0000e+00, -8.0466e-05,  2.2321e-03],
        [-8.0466e-05,  0.0000e+00,  5.9658e-04],
        [ 2.2321e-03,  5.9658e-04,  0.0000e+00]])

In [1]:
a = [1,2,3]
b = [4,5,6]

In [3]:
a+=b

In [4]:
a

[1, 2, 3, 4, 5, 6]